In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))

sys.path.append(base_path)


import numpy as np
import pandas as pd

import project_images_loader as pil
import utils.util as util
import utils.nifti_functions as nfc
import utils.functions as fc

import constants as cts
import utils.dfield_applier as dfield_applier
# from approach_images_similar.similarity_tables_reader import SimilarityTablesReader

from tqdm import tqdm

np.set_printoptions(precision=4)
features_loader_tissue_based = fc.load_module_from_path('features_loader_tissue_based', os.path.join(base_path, 'statistics/patients/inter_patient/tissue_based/features_loader.py'))

import itk

matching_ids = pil.get_matching_ids()


SERVER: False


In [2]:

def transform_segmentation_from_parameters_file(id_moving, obtained_pfile_path_name_list):
    param_obj = itk.ParameterObject.New()
    for obtained_pfile_path_name in obtained_pfile_path_name_list:
        param_obj.AddParameterFile(obtained_pfile_path_name)

    param_obj.SetParameter('FinalBSplineInterpolationOrder','0')

    seg_itk = itk.imread(pil.get_segmentation_path_name(3, id_moving, nb_tissues=3, absolute=True), itk.US)
    reg_seg = itk.transformix_filter(seg_itk, param_obj)
    return np.transpose(itk.array_from_image(reg_seg), (2, 1, 0)) 


def transform_image_from_parameters_file(id_moving, obtained_pfile_path_name_list):
    param_obj = itk.ParameterObject.New()
    for obtained_pfile_path_name in obtained_pfile_path_name_list:
        param_obj.AddParameterFile(obtained_pfile_path_name)

    img_itk = itk.imread(pil.get_registered_to_template_image_path_name(3, id_moving, pil.REGISTRATION_TO_MNI, absolute=True), itk.F)
    reg_img = itk.transformix_filter(img_itk, param_obj)
    return np.transpose(itk.array_from_image(reg_img), (2, 1, 0)) 

def get_first_similar_patient_id(id_target, exclude_ids=None):
    dice_mean_table_reder = features_loader_tissue_based.FeaturesLoader(3, matching_ids)
    patient_sim_table = dice_mean_table_reder.get_sorted_similarity_table(id_target, "dice_mean", 1, exclude_ids=exclude_ids)
    return patient_sim_table.iloc[0, 0]


In [3]:
val_images = ["08202"] + matching_ids[621:]
exclude_ids = val_images + pil.EXCLUDED_PATIENTS

id_target = val_images[0]
id_data = get_first_similar_patient_id(id_target, exclude_ids=exclude_ids)
print("Target: ", id_target, "Data: ", id_data)
img_trg_path_name = pil.get_registered_to_template_image_path_name(3, id_target, pil.REGISTRATION_TO_MNI, absolute=True)
seg_trg_path_name = pil.get_segmentation_path_name(3, id_target, nb_tissues=3, absolute=True)

img_data_path_name = pil.get_registered_to_template_image_path_name(3, id_data, pil.REGISTRATION_TO_MNI, absolute=True)
seg_data_path_name = pil.get_segmentation_path_name(3, id_data, nb_tissues=3, absolute=True)


img_trg, _ = nfc.load_nifti(img_trg_path_name)
seg_trg, _ = nfc.load_nifti(seg_trg_path_name)
img_trg = util.skull_stripping_and_normalize(img_trg, seg_trg)

img_data, _ = nfc.load_nifti(img_data_path_name)
seg_data, _ = nfc.load_nifti(seg_data_path_name)
img_data = util.skull_stripping_and_normalize(img_data, seg_data)

initial_metrics = util.get_metrics_dict(img_trg, img_data)
initial_dices = util.compute_multiple_dice_coefficients(seg_trg, seg_data)
print("Initial metrics: ", initial_metrics)
print("Initial dices: ", initial_dices)
# transform_image_from_parameters_file

for param_type in [cts.PARAMETERS_FILE_RIGID, cts.PARAMETERS_FILE_AFFINE, cts.PARAMETERS_FILE_BSPLINE_FAST, cts.PARAMETERS_FILE_BSPLINE]:
    # print("Transforming with ", param_type)
    img_data_reg = transform_image_from_parameters_file(id_data, cts.get_obtained_pfile_path_name_list(id_target, id_data, param_type))
    seg_data_reg = transform_segmentation_from_parameters_file(id_data, cts.get_obtained_pfile_path_name_list(id_target, id_data, param_type))
    img_data_reg = util.skull_stripping_and_normalize(img_data_reg, seg_data_reg)

    final_metrics = util.get_metrics_dict(img_trg, img_data_reg)
    final_dices = util.compute_multiple_dice_coefficients(seg_trg, seg_data_reg)
    print()
    print(f"{param_type} metrics: ", final_metrics)
    print(f"{param_type} dices: ", final_dices)




Target:  08202 Data:  13169
Initial metrics:  {'rms': 0.00013879164376406843, 'ncc': 0.5552947339915878, 'mi': 0.43915490121196576, 'ssim': 0.8497485547641972}
Initial dices:  {0: 0.980707540605286, 1: 0.6963545858296819, 2: 0.5915497613412489, 3: 0.6651192097320184}

RIG metrics:  {'rms': 0.000142168382581334, 'ncc': 0.5484092085878597, 'mi': 0.4392459043773216, 'ssim': 0.8500965865720409}
RIG dices:  {0: 0.9807721357294706, 1: 0.6963545858296819, 2: 0.5923856903205547, 3: 0.6657752666065169}

AFF metrics:  {'rms': 0.00013572551884614327, 'ncc': 0.5630579691914698, 'mi': 0.44547310957207475, 'ssim': 0.853133915420102}
AFF dices:  {0: 0.9813309880030436, 1: 0.6899812650548667, 2: 0.5973769685729584, 3: 0.6657740285663897}

BSP_FAST metrics:  {'rms': 9.777500333668423e-05, 'ncc': 0.779298726142277, 'mi': 0.5503773031979899, 'ssim': 0.9043758297880999}
BSP_FAST dices:  {0: 0.9869663574401795, 1: 0.8851998467628656, 2: 0.6996568811774817, 3: 0.7928249688901525}

BSP metrics:  {'rms': 7.44